In [0]:
spark.conf.set(
    "fs.azure.account.key.stretailadls01.dfs.core.windows.net",
    "account-key"
)

In [0]:
from pyspark.sql.functions import *

In [0]:
df_bronze = spark.read.format("delta")\
    .load("abfss://retail-project@stretailadls01.dfs.core.windows.net/bronze/sales/bronze-delta/")
df_bronze.show()

+-------+--------+-----------+----------+-------------+---------+--------+----------+------------+--------+------------+
|sale_id|store_id|customer_id| sale_date| product_name| category|quantity|unit_price|total_amount|discount|payment_mode|
+-------+--------+-----------+----------+-------------+---------+--------+----------+------------+--------+------------+
|      1|    S001|       C001|2024-01-01|         Rice|  Grocery|      10|        50|         500|       0|        Cash|
|      2|    S001|       C002|2024-01-01|Sunflower Oil|  Grocery|       5|       150|         750|      10|        Card|
|      3|    S001|       C003|2024-01-02|        Sugar|  Grocery|       8|        40|         320|       0|         UPI|
|      4|    S001|       C004|2024-01-02|   Surf Excel|Home Care|       3|       120|         360|       5|        Cash|
|      5|    S001|       C001|2024-01-03|         Rice|  Grocery|      15|        50|         750|       0|        Card|
|      6|    S002|       C002|20

In [0]:
display(df_bronze)

sale_id,store_id,customer_id,sale_date,product_name,category,quantity,unit_price,total_amount,discount,payment_mode
1,S001,C001,2024-01-01,Rice,Grocery,10,50,500,0,Cash
2,S001,C002,2024-01-01,Sunflower Oil,Grocery,5,150,750,10,Card
3,S001,C003,2024-01-02,Sugar,Grocery,8,40,320,0,UPI
4,S001,C004,2024-01-02,Surf Excel,Home Care,3,120,360,5,Cash
5,S001,C001,2024-01-03,Rice,Grocery,15,50,750,0,Card
6,S002,C002,2024-01-01,Rice,Grocery,20,50,1000,0,UPI
7,S002,C003,2024-01-01,Wheat,Grocery,15,45,675,5,Cash
8,S002,C004,2024-01-02,Sunflower Oil,Grocery,10,150,1500,10,Card
9,S002,C005,2024-01-02,Detergent,Home Care,8,90,720,0,UPI
10,S002,C001,2024-01-03,Sugar,Grocery,25,40,1000,0,Cash


In [0]:
df_clean =df_bronze.dropna(
            subset=["sale_id","store_id","customer_id","sale_date","product_name","total_amount"]
        )

In [0]:
display(df_bronze)

sale_id,store_id,customer_id,sale_date,product_name,category,quantity,unit_price,total_amount,discount,payment_mode
1,S001,C001,2024-01-01,Rice,Grocery,10,50,500,0,Cash
2,S001,C002,2024-01-01,Sunflower Oil,Grocery,5,150,750,10,Card
3,S001,C003,2024-01-02,Sugar,Grocery,8,40,320,0,UPI
4,S001,C004,2024-01-02,Surf Excel,Home Care,3,120,360,5,Cash
5,S001,C001,2024-01-03,Rice,Grocery,15,50,750,0,Card
6,S002,C002,2024-01-01,Rice,Grocery,20,50,1000,0,UPI
7,S002,C003,2024-01-01,Wheat,Grocery,15,45,675,5,Cash
8,S002,C004,2024-01-02,Sunflower Oil,Grocery,10,150,1500,10,Card
9,S002,C005,2024-01-02,Detergent,Home Care,8,90,720,0,UPI
10,S002,C001,2024-01-03,Sugar,Grocery,25,40,1000,0,Cash


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, DateType
from pyspark.sql.functions import col, to_date, current_timestamp

# Schema manually define చేయి
schema = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("store_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("sale_date", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("payment_mode", StringType(), True)
])

# Read with schema
df_sales = spark.read.format("csv")\
    .option("header", "true")\
    .schema(schema)\
    .load("abfss://retail-project@stretailadls01.dfs.core.windows.net/bronze/sales/")

# Clean చేయి
df_clean = df_sales.dropna(subset=["sale_id","store_id","sale_date","total_amount"])

df_clean = df_clean\
    .withColumn("sale_date", to_date(col("sale_date"), "yyyy-MM-dd"))\
    .withColumn("unit_price", col("unit_price").cast("decimal(10,2)"))\
    .withColumn("total_amount", col("total_amount").cast("decimal(10,2)"))\
    .withColumn("discount", col("discount").cast("decimal(5,2)"))

df_clean = df_clean.dropDuplicates(["sale_id"])
df_clean = df_clean.withColumn("ingested_at", current_timestamp())

print("Silver rows:", df_clean.count())
df_clean.show()

# Silver లో save చేయి
df_clean.write\
    .format("delta")\
    .mode("overwrite")\
    .save("abfss://retail-project@stretailadls01.dfs.core.windows.net/silver/sales/")

print("✅ Bronze to Silver — Sales Done!")

Silver rows: 15
+-------+--------+-----------+----------+-------------+---------+--------+----------+------------+--------+------------+--------------------+
|sale_id|store_id|customer_id| sale_date| product_name| category|quantity|unit_price|total_amount|discount|payment_mode|         ingested_at|
+-------+--------+-----------+----------+-------------+---------+--------+----------+------------+--------+------------+--------------------+
|     12|    S003|       C004|2024-01-02|         Rice|  Grocery|      30|     50.00|     1500.00|    0.00|         UPI|2026-04-20 04:00:...|
|     13|    S003|       C005|2024-01-02|        Wheat|  Grocery|      12|     45.00|      540.00|    5.00|        Cash|2026-04-20 04:00:...|
|     15|    S003|       C002|2024-01-03|    Detergent|Home Care|      10|     90.00|      900.00|   10.00|         UPI|2026-04-20 04:00:...|
|     11|    S003|       C003|2024-01-01|        Sugar|  Grocery|      25|     40.00|     1000.00|    0.00|        Card|2026-04-20 0

In [0]:
# ========== DIMENSIONS — Bronze to Silver ==========

# 1. Stores
df_stores = spark.read.format("csv")\
    .option("header", "true")\
    .load("abfss://retail-project@stretailadls01.dfs.core.windows.net/bronze/stores/")

df_stores = df_stores.dropDuplicates(["store_id"])\
    .dropna(subset=["store_id"])\
    .withColumn("ingested_at", current_timestamp())

df_stores.write.format("delta").mode("overwrite")\
    .save("abfss://retail-project@stretailadls01.dfs.core.windows.net/silver/dimensions/stores/")

print("✅ Stores Done!")

# 2. Products
df_products = spark.read.format("csv")\
    .option("header", "true")\
    .load("abfss://retail-project@stretailadls01.dfs.core.windows.net/bronze/products/")

df_products = df_products.dropDuplicates(["product_id"])\
    .dropna(subset=["product_id"])\
    .withColumn("ingested_at", current_timestamp())

df_products.write.format("delta").mode("overwrite")\
    .save("abfss://retail-project@stretailadls01.dfs.core.windows.net/silver/dimensions/products/")

print("✅ Products Done!")

# 3. Customers
df_customers = spark.read.format("csv")\
    .option("header", "true")\
    .load("abfss://retail-project@stretailadls01.dfs.core.windows.net/bronze/customers/")

df_customers = df_customers.dropDuplicates(["customer_id"])\
    .dropna(subset=["customer_id"])\
    .withColumn("ingested_at", current_timestamp())

df_customers.write.format("delta").mode("overwrite")\
    .save("abfss://retail-project@stretailadls01.dfs.core.windows.net/silver/dimensions/customers/")

print("✅ Customers Done!")

✅ Stores Done!
✅ Products Done!
✅ Customers Done!
